In [2]:
from pathlib import Path
import csv
import pandas as pd

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "Data" / "raw"

ddi = pd.read_csv(DATA_DIR / "db_drug_interactions.csv")
eg = pd.read_csv(DATA_DIR / "eg_drugs.csv")

In [3]:
#Cleaning and standardizing the names of active pharmaceutical ingredients.

#import libraries
import re
import pandas as pd

#delete the salts and standardize the names of active ingredients
SALTS = r"\b(hydrochloride|hcl|sodium|potassium|calcium|sulfate|sulphate|maleate|mesylate|tartrate|succinate|phosphate|acetate|citrate|fumarate|besylate)\b"
SYN = {"paracetamol": "acetaminophen", "adrenaline": "epinephrine", "salbutamol": "albuterol",
       "frusemide": "furosemide"}  

#clean and standardize the name of a single active ingredient.
def norm_ing(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"\(.*?\)", "", s)            
    s = re.sub(SALTS, "", s)
    s = re.sub(r"[^a-z0-9\- ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return SYN.get(s, s)

#handle a capsule or medication that may contain more than one active ingredient simultaneously 
def split_active(a):
    if pd.isna(a): return ()
    return tuple(sorted({norm_ing(x) for x in re.split(r"\s*\+\s*", a) if x.strip()}))

In [4]:
#Assessing the coverage of Egyptian drugs by the drug-drug interaction (DDI) database.

ddi_drugs = set(pd.concat([ddi["Drug 1"], ddi["Drug 2"]]).map(norm_ing))
eg["ingredients"] = eg["active"].map(split_active)

has_ing = eg[eg["ingredients"].map(len) > 0]
covered = has_ing["ingredients"].map(lambda t: all(i in ddi_drugs for i in t))
print("products fully covered by DDI vocabulary:", covered.mean())

#The most frequently occurring non-matching components—a candidate list for the SYN dictionary.
from collections import Counter
miss = Counter(i for t in has_ing["ingredients"] for i in t if i not in ddi_drugs)
print(miss.most_common(40))

products fully covered by DDI vocabulary: 0.45429339606788777
[('vitamin c', 1138), ('panthenol', 907), ('menthol', 602), ('vitamin d3', 505), ('olive oil', 422), ('vitamin b12', 421), ('tea tree oil', 413), ('vitamin b6', 361), ('aloe vera', 318), ('hyaluronic acid', 305), ('biotin', 301), ('chamomile', 281), ('camphor', 275), ('lactoferrin', 259), ('allantoin', 237), ('jojoba oil', 227), ('vitamin b1', 217), ('chlorhexidine', 215), ('eucalyptus oil', 211), ('clove oil', 210), ('collagen', 208), ('b6', 204), ('vit e', 198), ('vit c', 197), ('almond oil', 190), ('honey', 187), ('dimethicone', 182), ('wheat germ oil', 180), ('eucalyptus', 180), ('thyme', 180), ('b12', 179), ('peppermint oil', 174), ('vitamin d', 167), ('chamomile extract', 163), ('manganese', 161), ('aloe vera extract', 157), ('glucose', 154), ('chondroitin', 153), ('e', 148), ('shea butter', 148)]


In [5]:
#Converting DDI into a usable structure
#Processing and interpreting descriptive text regarding drug interactions, and converting it from unstructured text into structured data .
#identify the interaction type, direction, and pharmacokinetic or pharmacodynamic effect.

ddi["a"], ddi["b"] = ddi["Drug 1"].map(norm_ing), ddi["Drug 2"].map(norm_ing)
ddi["pair"] = [tuple(sorted(p)) for p in zip(ddi.a, ddi.b)]   

ATTR = re.compile(r"(serum concentration of the active metabolites|serum concentration|metabolism|excretion rate|excretion|absorption|therapeutic efficacy|risk or severity of adverse effects|bioavailability)", re.I)
DIRN = re.compile(r"\b(increased?|decreased?|reduced?)\b", re.I)
PD   = re.compile(r"may (increase|decrease) the ([\w\- ]+?) activities", re.I)

def parse(desc):
    m = PD.search(desc)
    if m:
        return pd.Series(["pharmacodynamic", m.group(1).lower(), m.group(2).lower()])
    a, d = ATTR.search(desc), DIRN.search(desc)
    return pd.Series([a.group(1).lower() if a else None,
                      ("increase" if d and d.group(1).lower().startswith("increas") else "decrease") if d else None,
                      None])
ddi[["attribute", "direction", "pd_effect"]] = ddi["Interaction Description"].apply(parse)
print(ddi["attribute"].isna().mean())   

0.035966190006317185


In [6]:
import numpy as np
rng = np.random.default_rng(42)
drugs = np.array(sorted(set(ddi.a) | set(ddi.b))); rng.shuffle(drugs)
test_drugs = set(drugs[: int(0.15 * len(drugs))])
a_t, b_t = ddi.a.isin(test_drugs), ddi.b.isin(test_drugs)
S3_test = ddi[a_t & b_t]; S2_test = ddi[a_t ^ b_t]; train = ddi[~a_t & ~b_t]